**Project:** Plant Disease Object Detection - YOLO11 Training Pipeline

**Course:** Final Project - Deep Learning

**Author:** Vu Huy Do

**Institution:** Phenikaa University

**Description:**
This script implements a state-of-the-art Object Detection benchmarking system:
1. Prepares the Google Colab environment and clones the PlantDoc dataset.
2. Parses bounding-box CSV files and converts them into normalized YOLO format.
3. Implements targeted data augmentations to mitigate class imbalance.
4. Trains a YOLO11m model to simultaneously locate and classify plant diseases.
5. Exports the highly-optimized model to ONNX format for edge deployment.

In [10]:
!pip install ultralytics pandas opencv-python onnx onnxruntime

In [11]:
import os
import sys
import json
import shutil
import pandas as pd
from PIL import Image
import yaml
import torch

# Ultralytics YOLO Configuration
try:
    from ultralytics import YOLO
except ImportError:
    print("[ERROR] Ultralytics framework is not installed. Please run: !pip install ultralytics")
    sys.exit(1)

### **CONFIGURATION SETTINGS**

In [12]:
class Config:
    PROJECT_DIR = "/content/drive/MyDrive/documents/DL-Projects/final-project"

    # Remote repository for the Object Detection variant of the PlantDoc dataset
    DATASET_REPO = "https://github.com/dovh11/PlantDoc-Object-Detection-Dataset.git"
    REPO_DIR = "./PlantDoc-Object-Detection-Dataset"

    # YOLO formatted dataset directory
    YOLO_DATA_DIR = "./yolo_plantdoc"

    # [UPDATE 1]: Increase the number of epochs to allow deeper model convergence
    EPOCHS = 200
    IMAGE_SIZE = 640
    BATCH_SIZE = 16

### **STEP 1: MOUNT GOOGLE DRIVE & SETUP REPOSITORY**

In [13]:
def setup_environment():
    print("\n--- STEP 1: Initializing Workspace and Environment ---")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("[SUCCESS] Google Drive successfully mounted.")
    except ImportError:
        print("[WARNING] Not running within Google Colab. Drive mounting bypassed.")

    os.makedirs(Config.PROJECT_DIR, exist_ok=True)

    if not os.path.exists(Config.REPO_DIR):
        print(f"[INFO] Cloning PlantDoc Object Detection repository from GitHub...")
        os.system(f"git clone {Config.DATASET_REPO}")
    else:
        print("[INFO] Dataset repository already exists in the current directory.")

### **STEP 2: YOLO FORMAT CONVERSION & DATASET RESTRUCTURING**

In [14]:
def convert_to_yolo_format():
    print("\n--- STEP 2: Converting CSV Annotations to Normalized YOLO Format ---")

    dirs_to_make = [
        f"{Config.YOLO_DATA_DIR}/images/train",
        f"{Config.YOLO_DATA_DIR}/images/val",
        f"{Config.YOLO_DATA_DIR}/labels/train",
        f"{Config.YOLO_DATA_DIR}/labels/val"
    ]

    # Clean directory if previously executed to prevent data corruption
    if os.path.exists(Config.YOLO_DATA_DIR):
        shutil.rmtree(Config.YOLO_DATA_DIR)

    for d in dirs_to_make:
        os.makedirs(d, exist_ok=True)

    train_csv_path = os.path.join(Config.REPO_DIR, 'train_labels.csv')
    test_csv_path = os.path.join(Config.REPO_DIR, 'test_labels.csv')

    try:
        train_df = pd.read_csv(train_csv_path)
        test_df = pd.read_csv(test_csv_path)
    except Exception as e:
        print(f"[ERROR] Failed to read annotation CSV files: {e}")
        sys.exit(1)

    # Extract globally unique classes across both splits
    all_classes = sorted(list(set(train_df['class'].unique()).union(set(test_df['class'].unique()))))
    class_to_id = {cls_name: i for i, cls_name in enumerate(all_classes)}

    print(f"[INFO] Identified {len(all_classes)} unique disease classes.")

    def process_split(df, source_img_dir, split_name):
        missing_images = 0
        successful_boxes = 0

        # Group operations by filename to handle multiple bounding boxes per image
        grouped = df.groupby('filename')

        for filename, group in grouped:
            src_img_path = os.path.join(source_img_dir, filename)

            if not os.path.exists(src_img_path):
                missing_images += 1
                continue

            # Verify image integrity to prevent dataloader crashes during training
            try:
                with Image.open(src_img_path) as img:
                    img.verify()
            except Exception:
                missing_images += 1
                continue

            dst_img_path = os.path.join(Config.YOLO_DATA_DIR, 'images', split_name, filename)
            shutil.copy(src_img_path, dst_img_path)

            txt_filename = os.path.splitext(filename)[0] + '.txt'
            txt_path = os.path.join(Config.YOLO_DATA_DIR, 'labels', split_name, txt_filename)

            with open(txt_path, 'w') as f:
                for _, row in group.iterrows():
                    img_w, img_h = row['width'], row['height']

                    # Dimensionality safeguard
                    if img_w <= 0 or img_h <= 0:
                        continue

                    xmin, ymin = row['xmin'], row['ymin']
                    xmax, ymax = row['xmax'], row['ymax']
                    cls_id = class_to_id[row['class']]

                    # Clamp coordinates to prevent out-of-bounds annotations
                    xmin, ymin = max(0, xmin), max(0, ymin)
                    xmax, ymax = min(img_w, xmax), min(img_h, ymax)

                    # Transform coordinates to YOLO constraints (center_x, center_y, w, h)
                    b_center_x = ((xmin + xmax) / 2.0) / img_w
                    b_center_y = ((ymin + ymax) / 2.0) / img_h
                    b_width    = (xmax - xmin) / img_w
                    b_height   = (ymax - ymin) / img_h

                    # Final normalization check
                    b_center_x = max(0.0, min(1.0, b_center_x))
                    b_center_y = max(0.0, min(1.0, b_center_y))
                    b_width    = max(0.0, min(1.0, b_width))
                    b_height   = max(0.0, min(1.0, b_height))

                    f.write(f"{cls_id} {b_center_x:.6f} {b_center_y:.6f} {b_width:.6f} {b_height:.6f}\n")
                    successful_boxes += 1

        print(f"[INFO] Split '{split_name}' processed: {successful_boxes} bounding boxes instantiated. {missing_images} corrupt files discarded.")

    process_split(train_df, os.path.join(Config.REPO_DIR, 'TRAIN'), 'train')
    process_split(test_df, os.path.join(Config.REPO_DIR, 'TEST'), 'val')

    # Generate YOLO configuration file
    yaml_path = os.path.join(Config.YOLO_DATA_DIR, 'dataset.yaml')
    yaml_content = {
        'path': os.path.abspath(Config.YOLO_DATA_DIR),
        'train': 'images/train',
        'val': 'images/val',
        'names': {i: name for i, name in enumerate(all_classes)}
    }

    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, sort_keys=False)

    print(f"[SUCCESS] Configuration file 'dataset.yaml' successfully generated at {yaml_path}")

    # Export dictionary map for inference API usage
    labels_json_path = os.path.join(Config.PROJECT_DIR, 'labels.json')
    with open(labels_json_path, 'w') as f:
        json.dump(all_classes, f, indent=4)
    print(f"[SUCCESS] Class mappings 'labels.json' safely exported to Google Drive.")

    return yaml_path, all_classes

### **STEP 3: YOLO TRAINING & DEPLOYMENT EXPORT ENGINE**

In [15]:
def train_and_export_yolo(yaml_path):
    print("\n--- STEP 3: Initializing Advanced Object Detection Training ---")

    # [UPDATE 2]: Upgrade to YOLO11m architecture to leverage advanced Attention mechanisms
    model = YOLO('yolo11m.pt')

    print(f"\n[INFO] Commencing training process for {Config.EPOCHS} epochs...")

    # [UPDATE 3]: Fine-tune Augmentation parameters and Early Stopping criteria
    results = model.train(
        data=yaml_path,
        epochs=Config.EPOCHS,
        imgsz=Config.IMAGE_SIZE,
        batch=Config.BATCH_SIZE,
        project=Config.PROJECT_DIR,
        name="plantdoc_yolo",
        exist_ok=True,
        patience=50,   # Extended patience threshold to prevent premature termination
        device=0 if torch.cuda.is_available() else 'cpu',

        # State-of-the-Art (SOTA) Augmentations customized for Phytopathology
        degrees=15.0,
        hsv_h=0.015,   # Hue variance to simulate different lighting conditions
        hsv_s=0.7,     # High saturation variance to handle diverse image qualities
        hsv_v=0.4,     # High brightness variance
        flipud=0.5,    # Enable vertical flipping (plant leaf orientation is invariant)
        mixup=0.2,     # Increase Mixup probability to blend multiple images
        copy_paste=0.3 # Increase Copy-Paste probability to synthesize rare disease spots on healthy backgrounds
    )

    print("\n[SUCCESS] Neural Network Training Successfully Completed!")

    print("\n--- STEP 4: Exporting Weights to ONNX for Live Deployment ---")

    best_weights_path = os.path.join(Config.PROJECT_DIR, "plantdoc_yolo", "weights", "best.pt")
    best_model = YOLO(best_weights_path)

    export_path = best_model.export(format='onnx', dynamic=False, opset=12)

    print(f"\n[SUCCESS] Highly-optimized ONNX model successfully exported to: {export_path}")
    print("\n=== DEPLOYMENT PIPELINE PREPARED: DOWNLOAD best.onnx & labels.json TO LOCAL MACHINE ===")

### **MAIN EXECUTION ROUTINE**

In [16]:
def main():
    setup_environment()
    yaml_path, classes = convert_to_yolo_format()
    train_and_export_yolo(yaml_path)

In [17]:
if __name__ == "__main__":
    main()


--- STEP 1: Initializing Workspace and Environment ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[SUCCESS] Google Drive successfully mounted.
[INFO] Dataset repository already exists in the current directory.

--- STEP 2: Converting CSV Annotations to Normalized YOLO Format ---
[INFO] Identified 29 unique disease classes.
[INFO] Split 'train' processed: 8454 bounding boxes instantiated. 3 corrupt files discarded.
[INFO] Split 'val' processed: 452 bounding boxes instantiated. 0 corrupt files discarded.
[SUCCESS] Configuration file 'dataset.yaml' successfully generated at ./yolo_plantdoc/dataset.yaml
[SUCCESS] Class mappings 'labels.json' safely exported to Google Drive.

--- STEP 3: Initializing Advanced Object Detection Training ---

[INFO] Commencing training process for 200 epochs...
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=Fal